# V127 - Corrected Top1 action selector, dual frozen encoders

Prioritas 1 dari audit `results/leader_audit/FINDINGS_AND_NEXT_MODEL.md`. Hipotesis: dua representasi frozen membantu memilih satu swap Top1 yang menguntungkan di antara rank 2-4. v124 tidak menguji ini dengan benar (label salah + fitur tak membedakan kandidat), jadi versi ini membangun ulang dengan benar: target delta swap yang diverifikasi identitasnya, fitur aksi yang berbeda per kandidat (termasuk interaksi embedding user-modul), analogue struktural champion yang tervalidasi nested sebagai pembanding, dua head kecil yang harus sepakat, maksimal satu swap per user, proteksi geometri gaya champion, dan gate tanpa penurunan setelah hasil terlihat. Gagal satu kunci berarti champion identik. Tanpa submission leaderboard.

In [1]:
import hashlib
import json
import os
import re
import time
from hashlib import sha256
from pathlib import Path

SMOKE = bool(int(os.getenv("V127_SMOKE", "0")))
SEED = 6161
N_SPLITS = 2 if SMOKE else 5
N_INNER = 2 if SMOKE else 3
N_EST = 20 if SMOKE else 150
GATE_FLOOR = 0.002
SLICE_FLOOR = 0.01
THR_GRID = [0.0, 0.0005, 0.001, 0.002]
E5_ID = "intfloat/multilingual-e5-base"
NOMIC_ID = "nomic-ai/nomic-embed-text-v2-moe"
EXP_TOP5 = "7c0d29a7309b7283a043b7d37ca5556ba7c0ca2270b0137a6e1e0878a69a5fe7"
EXP_FILE = "443785fb534df9820ce57bfe450a2768881625bf7ae37aa849dd0d9f2afc957d"
import numpy as np
import pandas as pd
rng = np.random.RandomState(SEED)

ROOT = Path("/home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender")
CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data"), Path("../data"), ROOT / "data"]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CH = next((p for p in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                       Path("results/v42/submission_v42_primary.csv"), ROOT / "results/v42/submission_v42_primary.csv"] if p.exists()), None)
assert CH is not None, "champion file not found"
assert hashlib.sha256(CH.read_bytes()).hexdigest() == EXP_FILE, "champion file changed"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ROOT / "results/v127")
OUT.mkdir(parents=True, exist_ok=True)
T = {}
print(f"SMOKE={SMOKE} DATA={DATA}")



SMOKE=False DATA=../data


## Load, audit inputs, provenance caches

Input SHA diverifikasi terhadap diagnostik audit. Assessment di-decode, di-clip ke [0,5] seperti parser champion, NaN diimputasi mean, plus hitungan sel invalid sebagai fitur. Cache embedding diverifikasi bentuk, urutan (probe indeks train/test), dan pipeline (re-encode 24 sampel, cosine > 0.999) sebelum dipakai; gagal berarti rebuild penuh ke direktori run ini, bukan file bersama.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
DISC = 1 / np.log2(np.arange(2, 7))
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
assert hashlib.sha256((DATA / "train_relevance.csv").read_bytes()).hexdigest() == "bf66eed0ad7a6d356d528dad317a32e9a46172e46995a16316c23023790ccc90"
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids
ch = pd.read_csv(CH)
assert ch["user_id"].tolist() == test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
Sraw = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
invalid = (Sraw < 0) | (Sraw > 5) | ~np.isfinite(Sraw)
print(f"invalid assessment cells: {invalid.sum()} users: {invalid.any(1).sum()}")
Sraw = np.clip(np.where(np.isfinite(Sraw), Sraw, np.nan), 0, 5)
Sraw = np.where(np.isnan(Sraw), np.nanmean(Sraw, axis=0), Sraw)
A = np.hstack([(Sraw - Sraw.mean(0)) / (Sraw.std(0) + 1e-9), invalid.sum(1, keepdims=True).astype(float)])
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
groups = np.array([re.sub(r"\d+", "#", " ".join(str(by_user[u][0]).lower().split()[:6])) if u in by_user else "NO_CHAT" for u in train["user_id"]])
strata = Y.argmax(1)
print(f"groups={len(set(groups))} train={n_train} test={len(test_ids)}")

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)[:, :5]
    rel = np.take_along_axis(yt, order, 1)
    dcg = (rel * DISC).sum(1)
    ideal = (np.sort(yt, axis=1)[:, -5:][:, ::-1] * DISC).sum(1)
    return dcg / np.maximum(ideal, 1e-12)

import torch
from sentence_transformers import SentenceTransformer
E5C = ROOT / "results/v58/e5_users.npy"
NOC = ROOT / "results/v124/nomic_users.npy"
assert E5C.exists() and NOC.exists(), "shared embedding caches missing"
E5U = np.asarray(np.load(E5C), dtype=np.float32)
NOU = np.asarray(np.load(NOC), dtype=np.float32)
assert E5U.shape == (5000, 768) and NOU.shape == (5000, 768), (E5U.shape, NOU.shape)
t0 = time.time()
probe_idx = [0, 7, 3999, 4000, 4999]
e5t = SentenceTransformer(E5_ID)
pe = np.asarray(e5t.encode(["query: " + docs[i] for i in probe_idx], normalize_embeddings=True))
ce = (pe * E5U[probe_idx]).sum(1)
print(f"e5 probe cosine: {np.round(ce, 6)}")
assert (ce > 0.999).all(), "e5 cache pipeline mismatch"
del e5t
torch.cuda.empty_cache()
from transformers import AutoModel, AutoTokenizer
import torch.nn.functional as F
nTok = AutoTokenizer.from_pretrained(NOMIC_ID, trust_remote_code=True)
nNet = AutoModel.from_pretrained(NOMIC_ID, trust_remote_code=True, torch_dtype=torch.float16, device_map={"": 0} if torch.cuda.is_available() else None).eval()
pn = []
with torch.no_grad():
    for i in probe_idx:
        e = nTok("search_query: " + docs[i], padding=True, truncation=True, max_length=256, return_tensors="pt")
        e = {k: v.to(nNet.device) for k, v in e.items()}
        h = nNet(**e).last_hidden_state
        m = e["attention_mask"].unsqueeze(-1).float()
        v = ((h * m).sum(1) / m.sum(1).clamp_min(1)).float()
        pn.append((v / v.norm()).cpu().numpy().ravel())
pn = np.array(pn)
cn = (pn * (NOU[probe_idx] / (np.linalg.norm(NOU[probe_idx], axis=1, keepdims=True) + 1e-12))).sum(1)
print(f"nomic probe cosine: {np.round(cn, 6)}")
assert (cn > 0.999).all(), "nomic cache pipeline mismatch"
del nNet
torch.cuda.empty_cache()
T["probe"] = round(time.time() - t0, 1)
print(f"provenance OK in {T['probe']}s")



invalid assessment cells: 251 users: 73
groups=290 train=4000 test=1000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

e5 probe cosine: [1. 1. 1. 1. 1.]


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


/home/athilla/.cache/huggingface/modules/transformers_modules/nomic_hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning: Install Nomic's megablocks fork for better speed: `pip install git+https://github.com/nomic-ai/megablocks.git`
  warnings.warn("Install Nomic's megablocks fork for better speed: " +


[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


nomic probe cosine: [1.       0.999999 1.       1.       0.999999]
provenance OK in 36.3s


## Syllabus embeddings and nested champion analogue

Silabus di-encode kedua encoder lalu disimpan di direktori run. Analogue: Ridge word/char (vocab fit per outer-train) lalu kalibrator LightGBM per modul di atas inner-OOF plus one-hot top1 inner sebagai proxy intent, lalu blend konsensus berbobot OOF-mean outer-train. Seluruh supervised nested; tidak ada intent bundle Hybrid/B-rule (dicatat sebagai deviasi).

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
import lightgbm as lgb
syll = [str(catalog.set_index("module_id").loc[m, "description_and_syllabus"]) if m in set(catalog["module_id"]) else "" for m in MODS]
t0 = time.time()
e5t = SentenceTransformer(E5_ID)
ES = np.asarray(e5t.encode(["passage: " + t for t in syll], normalize_embeddings=True))
del e5t
torch.cuda.empty_cache()
nTok = AutoTokenizer.from_pretrained(NOMIC_ID, trust_remote_code=True)
nNet = AutoModel.from_pretrained(NOMIC_ID, trust_remote_code=True, torch_dtype=torch.float16, device_map={"": 0} if torch.cuda.is_available() else None).eval()
NS = []
with torch.no_grad():
    for s in syll:
        e = nTok("search_document: " + s, padding=True, truncation=True, max_length=256, return_tensors="pt")
        e = {k: v.to(nNet.device) for k, v in e.items()}
        h = nNet(**e).last_hidden_state
        m = e["attention_mask"].unsqueeze(-1).float()
        v = ((h * m).sum(1) / m.sum(1).clamp_min(1)).float()
        NS.append((v / v.norm()).cpu().numpy().ravel())
NS = np.array(NS, dtype=np.float32)
del nNet
torch.cuda.empty_cache()
np.save(OUT / "e5_syll.npy", ES)
np.save(OUT / "nomic_syll.npy", NS)
T["syll"] = round(time.time() - t0, 1)
print(f"syllabi encoded {ES.shape} {NS.shape} in {T['syll']}s")

t0 = time.time()
ANCH = np.zeros_like(Y)
fids = np.zeros(n_train, int)
outer = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(outer.split(docs[:n_train], strata, groups)):
    Vw = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit([docs[u] for u in tr])
    Vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=8000, min_df=3).fit([docs[u] for u in tr])
    Xall = np.hstack([Vw.transform(docs).toarray(), Vc.transform(docs).toarray()])
    Xtr, Xva = Xall[tr], Xall[va]
    stage = np.zeros((len(va), len(MODS)))
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(Xtr, Y[tr, j])
        stage[:, j] = m.predict(Xva)
    inOOF = np.zeros((len(tr), len(MODS)))
    inner = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=SEED + fi).split(Xtr, Y[tr].argmax(1), groups[tr])
    for tr2, va2 in inner:
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(Xtr[tr2], Y[tr][tr2, j])
            inOOF[va2, j] = m.predict(Xtr[va2])
    inTop = np.zeros_like(inOOF)
    inTop[np.arange(len(tr)), inOOF.argmax(1)] = 1.0
    vaTop = np.zeros((len(va), len(MODS)))
    vaTop[np.arange(len(va)), stage.argmax(1)] = 1.0
    cal = np.zeros((len(va), len(MODS)))
    for j in range(len(MODS)):
        g = lgb.LGBMRegressor(objective="regression", n_estimators=N_EST, learning_rate=0.05,
                                num_leaves=31, min_data_in_leaf=50, verbose=-1, deterministic=True, seed=SEED)
        g.fit(np.hstack([inOOF, inTop]), Y[tr, j])
        cal[:, j] = g.predict(np.hstack([stage, vaTop]))
    ANCH[va] = 0.5 * stage + 0.5 * cal
    fids[va] = fi + 1
    print(f"outer {fi + 1} analogue OOF={ndcg5(Y[va], ANCH[va]).mean():.5f}", flush=True)
T["analogue"] = round(time.time() - t0, 1)
print(f"ANALOGUE OOF={ndcg5(Y, ANCH).mean():.5f} (fixed 0.5/0.5 consensus, mirrors frozen v42 weights)")






Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/home/athilla/.cache/huggingface/modules/transformers_modules/nomic_hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning: Install Nomic's megablocks fork for better speed: `pip install git+https://github.com/nomic-ai/megablocks.git`
  warnings.warn("Install Nomic's megablocks fork for better speed: " +


syllabi encoded (17, 768) (17, 768) in 53.9s


outer 1 analogue OOF=0.64334


outer 2 analogue OOF=0.65668


outer 3 analogue OOF=0.63491


outer 4 analogue OOF=0.57833


outer 5 analogue OOF=0.59630


ANALOGUE OOF=0.61556 (fixed 0.5/0.5 consensus, mirrors frozen v42 weights)


# Counterfactual labels with verified identity

Target aksi memakai rumus audit: selisih relevansi challenger vs incumbent kali selisih discount,
dibagi IDCG user. Tiga verifikasi: (a) 300 user acak dibuktikan sama dengan scoring swap skor
langsung pada toleransi 1e-9; (b) user dengan label berbeda antar aksi harus > 0 (jebakan v124);
(c) incumbent di sini adalah analogue OOF, bukan Ridge sederhana.


In [4]:
t0 = time.time()
base_order = np.argsort(-ANCH, axis=1, kind="stable")
base_ndcg = ndcg5(Y, ANCH)
ideal = (np.sort(Y, axis=1)[:, -5:][:, ::-1] * DISC).sum(1)
chal = np.zeros((n_train, 3), int)
delta = np.zeros((n_train, 3))
for u in range(n_train):
    for k, r in enumerate((1, 2, 3)):
        ci = int(base_order[u, r])
        ii = int(base_order[u, 0])
        chal[u, k] = ci
        delta[u, k] = (Y[u, ci] - Y[u, ii]) * (DISC[0] - DISC[r]) / max(ideal[u], 1e-12)
ndiff = int((np.ptp(delta, axis=1) > 1e-12).sum())
assert ndiff > 0, "v124 trap: all actions identical"
print(f"users with differing action labels: {ndiff} / {n_train}")
print(f"mean utility={delta.mean():+.6f} positive-action frac={(delta > 0).mean():.3f}")
rs = np.random.RandomState(0).choice(n_train, 300, replace=False)
for u in rs:
    for k, r in enumerate((1, 2, 3)):
        sc = ANCH[u].copy()
        sc[base_order[u, 0]], sc[base_order[u, r]] = sc[base_order[u, r]], sc[base_order[u, 0]]
        np.testing.assert_allclose(ndcg5(Y[u:u+1], sc.reshape(1, -1))[0] - base_ndcg[u], delta[u, k], atol=1e-9)
print("verified against direct score-swap scoring on 300 users")
T["labels"] = round(time.time() - t0, 1)



users with differing action labels: 3953 / 4000
mean utility=-0.085485 positive-action frac=0.208
verified against direct score-swap scoring on 300 users


# Action features, heads, policy, gates

Fitur control (margin/rank/gap anchor + assessment) vs full (tambah interaksi embedding user-modul
per encoder). Tiga head Ridge kecil: control, e5, nomic. Threshold dari grid tetap dipilih di inner
split outer-train (tie mengarah ke nilai lebih besar). Kebijakan: dua head embedding sepakat pada
kandidat yang sama dan keduanya melewati threshold; control dipakai hanya untuk mengukur tambahan
manfaat representasi dengan aturan satu-head yang sama. Maksimal satu swap per user, selebihnya
abstain. Deployment memakai permutasi nilai skor anchor Top-4 gaya v42: Top5 dan rank5 dijamin tetap.


In [5]:
t0 = time.time()
E5u = E5U[:n_train].astype(np.float64)
NOu = NOU[:n_train].astype(np.float64)
E5u /= (np.linalg.norm(E5u, axis=1, keepdims=True) + 1e-12)
NOu /= (np.linalg.norm(NOu, axis=1, keepdims=True) + 1e-12)
E5s = (ES / (np.linalg.norm(ES, axis=1, keepdims=True) + 1e-12)).astype(np.float64)
NOs = (NS / (np.linalg.norm(NS, axis=1, keepdims=True) + 1e-12)).astype(np.float64)
readiness = A[:n_train, :-1].mean(1)
invcount = A[:n_train, -1]
anch_margin = np.ptp(ANCH[np.arange(n_train)[:, None], base_order[:, :2]], 1)
FC, FF = 5, 8
def build_features(users, order_u, chal_u, anchor_ref):
    nu = len(users)
    rank = np.tile([1, 2, 3], (nu, 1))
    gap = anchor_ref[users[:, None], chal_u] - anchor_ref[users[:, None], order_u[:, :1]]
    ctrl = np.stack([np.repeat(anch_margin[users, None], 3, 1), rank, gap,
                     np.repeat(readiness[users, None], 3, 1), np.repeat(invcount[users, None], 3, 1)], -1)
    eu, ou = E5u[users], NOu[users]
    fe = np.concatenate([ctrl] + [np.stack([(eu * (E5s[chal_u[:, k]] - E5s[order_u[:, 0]])).sum(1) for k in range(3)], 1)[..., None], np.stack([(eu * E5s[chal_u[:, k]]).sum(1) for k in range(3)], 1)[..., None], np.stack([(eu * E5s[order_u[:, 0]]).sum(1) for _ in range(3)], 1)[..., None]], -1)
    fo = np.concatenate([ctrl] + [np.stack([(ou * (NOs[chal_u[:, k]] - NOs[order_u[:, 0]])).sum(1) for k in range(3)], 1)[..., None], np.stack([(ou * NOs[chal_u[:, k]]).sum(1) for k in range(3)], 1)[..., None], np.stack([(ou * NOs[order_u[:, 0]]).sum(1) for _ in range(3)], 1)[..., None]], -1)
    assert (np.ptp(fe.reshape(nu, -1), 1) > 0).all() and (np.ptp(fo.reshape(nu, -1), 1) > 0).all()
    return ctrl, fe, fo
CTRL_ALL, FE_ALL, FO_ALL = build_features(np.arange(n_train), base_order, chal, ANCH)
print(f"control dim={CTRL_ALL.shape[2]} full dim={FE_ALL.shape[2]}")

def deploy_row(anchor_row, order_row, rpos):
    assert rpos in (1, 2, 3)
    oo = list(order_row[:4])
    oo[0], oo[rpos] = oo[rpos], oo[0]
    vals = sorted([anchor_row[i] for i in order_row[:4]], reverse=True)
    out = anchor_row.copy()
    for pos, mod in enumerate(oo):
        out[mod] = vals[pos]
    return out
T["feats"] = round(time.time() - t0, 1)
print("feature builders and deploy rule ready")





control dim=5 full dim=8
feature builders and deploy rule ready


# Outer folds: heads, threshold, policy, gates

Setiap outer fold: head Ridge (control, e5, nomic) dilatih di baris aksi outer-train, threshold
dipilih di inner split outer-train dari grid tetap (tie ke nilai lebih besar), lalu kebijakan
diterapkan di outer-val. Deployment hanya untuk user terpilih; baris lain identik anchor.
Setiap baris ter-deploy diverifikasi Top5 dan rank5 tetap. Gate: FULL vs ANCHOR; tambahan manfaat
representasi diukur FULL vs CONTROL dengan prosedur yang sama.


In [6]:
t0 = time.time()
splits = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(docs[:n_train], strata, groups))
thr_cand = np.array(THR_GRID)
rows, FINC = [], ANCH.copy()
FINF = ANCH.copy()
deploy_count_C = np.zeros(n_train, int)
deploy_count_F = np.zeros(n_train, int)
thrF_pick, thrC_pick = [], []

def apply_single(uu, U, thr):
    F = ANCH[uu].copy()
    n = 0
    for j, u in enumerate(uu):
        a = int(U[j].argmax())
        if U[j, a] > thr:
            F[j] = deploy_row(ANCH[u], base_order[u], a + 1)
            n += 1
    return F, n

def apply_agree(uu, U1, U2, thr):
    F = ANCH[uu].copy()
    n = 0
    for j, u in enumerate(uu):
        a1, a2 = int(U1[j].argmax()), int(U2[j].argmax())
        if a1 == a2 and U1[j, a1] > thr and U2[j, a2] > thr:
            F[j] = deploy_row(ANCH[u], base_order[u], a1 + 1)
            n += 1
    return F, n

def gain(F, uu):
    return float(ndcg5(Y[uu], F).mean() - ndcg5(Y[uu], ANCH[uu]).mean())

def arows(ii):
    return (ii[:, None] * 3 + np.arange(3)).ravel()

for fi, (tr, va) in enumerate(splits):
    assert (fids[va] == fi + 1).all()
    Xc_tr = CTRL_ALL[tr].reshape(-1, FC)
    Xe_tr = FE_ALL[tr].reshape(-1, FF)
    Xo_tr = FO_ALL[tr].reshape(-1, FF)
    d_tr = delta[tr].reshape(-1)
    iu = list(StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED + fi).split(tr, strata[tr]))
    gC = np.mean([[gain(apply_single(tr[ib], Ridge(alpha=3.0).fit(Xc_tr[arows(ia)], d_tr[arows(ia)]).predict(CTRL_ALL[tr[ib]].reshape(-1, FC)).reshape(-1, 3), thr)[0], tr[ib]) for thr in thr_cand] for ia, ib in iu], 0)
    thrC = float(thr_cand[np.where(gC == gC.max())[0][-1]])
    gF = np.mean([[gain(apply_agree(tr[ib], Ridge(alpha=3.0).fit(Xe_tr[arows(ia)], d_tr[arows(ia)]).predict(FE_ALL[tr[ib]].reshape(-1, FF)).reshape(-1, 3), Ridge(alpha=3.0).fit(Xo_tr[arows(ia)], d_tr[arows(ia)]).predict(FO_ALL[tr[ib]].reshape(-1, FF)).reshape(-1, 3), thr)[0], tr[ib]) for thr in thr_cand] for ia, ib in iu], 0)
    thrF = float(thr_cand[np.where(gF == gF.max())[0][-1]])
    thrC_pick.append(thrC)
    thrF_pick.append(thrF)
    Uc_va = Ridge(alpha=3.0).fit(Xc_tr, d_tr).predict(CTRL_ALL[va].reshape(-1, FC)).reshape(-1, 3)
    Ue_va = Ridge(alpha=3.0).fit(Xe_tr, d_tr).predict(FE_ALL[va].reshape(-1, FF)).reshape(-1, 3)
    Un_va = Ridge(alpha=3.0).fit(Xo_tr, d_tr).predict(FO_ALL[va].reshape(-1, FF)).reshape(-1, 3)
    FCv, nCv = apply_single(va, Uc_va, thrC)
    FFv, nFv = apply_agree(va, Ue_va, Un_va, thrF)
    FINC[va], FINF[va] = FCv, FFv
    deploy_count_C[va] = (np.abs(FCv - ANCH[va]).sum(1) > 0).astype(int)
    deploy_count_F[va] = (np.abs(FFv - ANCH[va]).sum(1) > 0).astype(int)
    rows.append({"fold": fi, "thrC": thrC, "thrF": thrF, "nC": int(nCv), "nF": int(nFv),
                 "oof_anchor": float(ndcg5(Y[va], ANCH[va]).mean()),
                 "oof_ctrl": float(ndcg5(Y[va], FCv).mean()), "oof_full": float(ndcg5(Y[va], FFv).mean())})
OOF = pd.DataFrame(rows)
print(OOF.to_string(index=False))
for u in np.where(deploy_count_F > 0)[0][:2000]:
    o, f = np.argsort(-ANCH[u], kind="stable"), np.argsort(-FINF[u], kind="stable")
    assert set(o[:5]) == set(f[:5]) and o[4] == f[4]
for u in np.where(deploy_count_F == 0)[0][:2000]:
    assert (FINF[u] == ANCH[u]).all()
print("deploy geometry verified: Top5 and rank5 preserved; abstained rows identical")
T["folds"] = round(time.time() - t0, 1)






 fold  thrC   thrF  nC  nF  oof_anchor  oof_ctrl  oof_full
    0 0.001 0.0020   3   6    0.643336  0.643309  0.642884
    1 0.000 0.0020   5   4    0.656684  0.657083  0.656743
    2 0.002 0.0005   3  21    0.634913  0.635091  0.635986
    3 0.001 0.0005   4   6    0.578331  0.578457  0.578033
    4 0.002 0.0020   2   2    0.596298  0.596159  0.596159
deploy geometry verified: Top5 and rank5 preserved; abstained rows identical


# Metrics, gates, test parity, files

Estimand berbobot user, CI cluster prefix group, minimum blok modul-ID dominan, dan irisan no-chat.
Test memakai rantai yang sama: refit penuh, threshold dipilih ulang di inner split train, fitur
paritas, satu kebijakan. Lolos gate menulis kandidat; gagal menulis salinan champion yang
terverifikasi identik peringkat. Tidak ada klaim menang dari smoke.


In [7]:
t0 = time.time()
nC = ndcg5(Y, FINC)
nF = ndcg5(Y, FINF)
nA = ndcg5(Y, ANCH)
dC, dF = nC - nA, nF - nA
rng = np.random.RandomState(SEED)
uniq, inv = np.unique(groups, return_inverse=True)
sS = np.bincount(inv, weights=dF)
sZ = np.bincount(inv)
dr = rng.randint(len(uniq), size=(2000, len(uniq)))
ciF = np.quantile(sS[dr].sum(1) / sZ[dr].sum(1), [0.025, 0.975])
sSC = np.bincount(inv, weights=dC)
ciC = np.quantile(sSC[dr].sum(1) / sZ[dr].sum(1), [0.025, 0.975])
foldF = np.array([dF[fids == f].mean() for f in range(1, N_SPLITS + 1)])
foldC = np.array([dC[fids == f].mean() for f in range(1, N_SPLITS + 1)])
top1 = base_order[:, 0]
blk = {int(m): float(dF[top1 == m].mean()) for m in np.unique(top1)}
dom = max(blk, key=lambda m: int((top1 == m).sum()))
nchat = np.array([len(d) == 0 for d in docs[:n_train]])
print(f"pooled FULL {dF.mean():+.5f} CI[{ciF[0]:+.5f},{ciF[1]:+.5f}] worst-fold {foldF.min():+.5f}")
print(f"pooled CTRL {dC.mean():+.5f} CI[{ciC[0]:+.5f},{ciC[1]:+.5f}] worst-fold {foldC.min():+.5f}")
print(f"dominant Top1-module block M_{dom:03d} (n={(top1 == dom).sum()}) gain {blk[dom]:+.5f}; min block {min(blk.values()):+.5f}")
print(f"no-chat slice (n={nchat.sum()}) FULL gain {dF[nchat].mean() if nchat.sum() else float('nan'):+.5f}")
print(f"deployed FULL {(deploy_count_F > 0).sum()} users, CTRL {(deploy_count_C > 0).sum()} users; thrF per-fold {thrF_pick}")
adopt = bool(dF.mean() >= GATE_FLOOR and (foldF >= 0).all() and ciF[0] > 0 and blk[dom] >= 0)
print("ADOPT:" , adopt)
T["metrics"] = round(time.time() - t0, 1)

t0 = time.time()
n_test = len(test_ids)
NM = len(MODS)
VWf = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit(docs[:n_train])
VCf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=8000, min_df=3).fit(docs[:n_train])
XtrA = np.hstack([VWf.transform(docs[:n_train]).toarray(), VCf.transform(docs[:n_train]).toarray()])
XteA = np.hstack([VWf.transform(docs[n_train:]).toarray(), VCf.transform(docs[n_train:]).toarray()])
stage_te = np.zeros((n_test, NM))
for j in range(NM):
    m = Ridge(alpha=1.0)
    m.fit(XtrA, Y[:, j])
    stage_te[:, j] = m.predict(XteA)
inOOF = np.zeros((n_train, NM))
innerF = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=SEED).split(docs[:n_train], strata, groups)
for tr2, va2 in innerF:
    st2 = np.zeros((len(va2), NM))
    for j in range(NM):
        m = Ridge(alpha=1.0)
        m.fit(XtrA[tr2], Y[tr2, j])
        st2[:, j] = m.predict(XtrA[va2])
    inOOF[va2] = st2
inTop = np.zeros_like(inOOF)
inTop[np.arange(n_train), inOOF.argmax(1)] = 1.0
teTop = np.zeros((n_test, NM))
teTop[np.arange(n_test), stage_te.argmax(1)] = 1.0
P_te = np.zeros((n_test, NM))
for j in range(NM):
    g = lgb.LGBMRegressor(objective="regression", n_estimators=N_EST, learning_rate=0.05,
                          num_leaves=31, min_data_in_leaf=50, verbose=-1, deterministic=True, seed=SEED)
    g.fit(np.hstack([inOOF, inTop]), Y[:, j])
    P_te[:, j] = g.predict(np.hstack([stage_te, teTop]))
ANCH_te = 0.5 * stage_te + 0.5 * P_te
order_te = np.argsort(-ANCH_te, axis=1, kind="stable")
chal_te = order_te[:, 1:4]
E5t = E5U[n_train:].astype(np.float64)
NOt = NOU[n_train:].astype(np.float64)
E5t /= (np.linalg.norm(E5t, axis=1, keepdims=True) + 1e-12)
NOt /= (np.linalg.norm(NOt, axis=1, keepdims=True) + 1e-12)
readme_t = A[n_train:, :-1].mean(1)
invc_t = A[n_train:, -1]
marg_t = np.ptp(ANCH_te[np.arange(n_test)[:, None], order_te[:, :2]], 1)
rank3 = np.tile([1, 2, 3], (n_test, 1))
gap_t = ANCH_te[np.arange(n_test)[:, None], chal_te] - ANCH_te[np.arange(n_test)[:, None], order_te[:, :1]]
ctlt = np.stack([np.repeat(marg_t[:, None], 3, 1), rank3, gap_t,
                 np.repeat(readme_t[:, None], 3, 1), np.repeat(invc_t[:, None], 3, 1)], -1)
fete = np.concatenate([ctlt, np.stack([(E5t * (E5s[chal_te[:, k]] - E5s[order_te[:, 0]])).sum(1) for k in range(3)], 1)[..., None], np.stack([(E5t * E5s[chal_te[:, k]]).sum(1) for k in range(3)], 1)[..., None], np.stack([(E5t * E5s[order_te[:, 0]]).sum(1) for _ in range(3)], 1)[..., None]], -1)
fotn = np.concatenate([ctlt, np.stack([(NOt * (NOs[chal_te[:, k]] - NOs[order_te[:, 0]])).sum(1) for k in range(3)], 1)[..., None], np.stack([(NOt * NOs[chal_te[:, k]]).sum(1) for k in range(3)], 1)[..., None], np.stack([(NOt * NOs[order_te[:, 0]]).sum(1) for _ in range(3)], 1)[..., None]], -1)
assert fete.shape == (n_test, 3, FF) and fotn.shape == (n_test, 3, FF)
assert (np.ptp(fete.reshape(n_test, -1), 1) > 0).all() and (np.ptp(fotn.reshape(n_test, -1), 1) > 0).all()
XcA, XeA, XoA = CTRL_ALL.reshape(-1, FC), FE_ALL.reshape(-1, FF), FO_ALL.reshape(-1, FF)
dA = delta.reshape(-1)
hE = Ridge(alpha=3.0).fit(XeA, dA)
hN = Ridge(alpha=3.0).fit(XoA, dA)
iu2 = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED).split(np.arange(n_train), strata))
gFt = []
for thr in thr_cand:
    gg = []
    for ia, ib in iu2:
        V1 = Ridge(alpha=3.0).fit(XeA[arows(ia)], dA[arows(ia)]).predict(FE_ALL[ib].reshape(-1, FF)).reshape(-1, 3)
        V2 = Ridge(alpha=3.0).fit(XoA[arows(ia)], dA[arows(ia)]).predict(FO_ALL[ib].reshape(-1, FF)).reshape(-1, 3)
        F_ib, _ = apply_agree(ib, V1, V2, thr)
        gg.append(gain(F_ib, ib))
    gFt.append(float(np.mean(gg)))
thrT = float(thr_cand[np.where(np.array(gFt) == np.max(gFt))[0][-1]])
Ue_t = hE.predict(fete.reshape(-1, FF)).reshape(-1, 3)
Un_t = hN.predict(fotn.reshape(-1, FF)).reshape(-1, 3)
FIN_te = ANCH_te.copy()
nact = 0
for j in range(n_test):
    ae, an = int(Ue_t[j].argmax()), int(Un_t[j].argmax())
    if ae == an and Ue_t[j, ae] > thrT and Un_t[j, an] > thrT:
        FIN_te[j] = deploy_row(ANCH_te[j], order_te[j], ae + 1)
        nact += 1
for u in range(n_test):
    o, f = np.argsort(-ANCH_te[u], kind="stable"), np.argsort(-FIN_te[u], kind="stable")
    assert set(o[:5]) == set(f[:5]) and o[4] == f[4]
print(f"test thr={thrT} actions={nact}/{n_test}; test deploy geometry verified")
T["test"] = round(time.time() - t0, 1)
t0 = time.time()
OUT.mkdir(parents=True, exist_ok=True)
REPORT = {"seed": SEED, "splits": N_SPLITS, "adopt": adopt, "thrF": thrF_pick, "thrC": thrC_pick, "thr_test": thrT,
          "pooled_full": float(dF.mean()), "ci_full": [float(ciF[0]), float(ciF[1])], "worst_fold_full": float(foldF.min()),
          "pooled_ctrl": float(dC.mean()), "ci_ctrl": [float(ciC[0]), float(ciC[1])], "worst_fold_ctrl": float(foldC.min()),
          "fold_table": OOF.to_dict("records"), "dominant_block": [dom, float(blk[dom])], "min_block": float(min(blk.values())),
          "test_actions": int(nact), "smoke": SMOKE, "timings": T,
          "cache_sha": {"e5": sha256(E5U.tobytes()).hexdigest()[:16], "nomic": sha256(NOU.tobytes()).hexdigest()[:16]}}
(OUT / "v127_gate_report.json").write_text(json.dumps(REPORT, indent=1))
assert len(test_ids) == n_test
if adopt:
    sub = pd.DataFrame({"user_id": test_ids})
    for j, m in enumerate(MODS):
        sub[m] = FIN_te[:, j]
    sub.to_csv(OUT / "candidate_v127_primary.csv", index=False)
    print("wrote candidate_v127_primary.csv")
else:
    (OUT / "champion_unchanged_v127.csv").write_bytes(CH.read_bytes())
    assert sha256((OUT / "champion_unchanged_v127.csv").read_bytes()).hexdigest() == EXP_FILE
    print("gate failed: champion byte-copy written, hash-verified identical")
(OUT / "NOTES.md").write_text(f"# V127\nadopt={adopt} pooled_full={dF.mean():+.5f} CI=[{ciF[0]:+.5f},{ciF[1]:+.5f}] worst_fold={foldF.min():+.5f} dom_block_gain={blk[dom]:+.5f} test_actions={nact} thr_test={thrT} smoke={SMOKE} timings={T}\n")
print(json.dumps(REPORT, indent=1)[:1500])







pooled FULL +0.00002 CI[-0.00035,+0.00041] worst-fold -0.00045
pooled CTRL +0.00007 CI[-0.00007,+0.00023] worst-fold -0.00014
dominant Top1-module block M_002 (n=751) gain -0.00019; min block -0.00071
no-chat slice (n=117) FULL gain +0.00000
deployed FULL 39 users, CTRL 17 users; thrF per-fold [0.002, 0.002, 0.0005, 0.0005, 0.002]
ADOPT: False


test thr=0.002 actions=15/1000; test deploy geometry verified
gate failed: champion byte-copy written, hash-verified identical
{
 "seed": 6161,
 "splits": 5,
 "adopt": false,
 "thrF": [
  0.002,
  0.002,
  0.0005,
  0.0005,
  0.002
 ],
 "thrC": [
  0.001,
  0.0,
  0.002,
  0.001,
  0.002
 ],
 "thr_test": 0.002,
 "pooled_full": 1.6232642081740966e-05,
 "ci_full": [
  -0.00035196791675873567,
  0.00041260894021458817
 ],
 "worst_fold_full": -0.0004526480721122998,
 "pooled_ctrl": 7.290213203275005e-05,
 "ci_ctrl": [
  -6.941996891144598e-05,
  0.0002333633094888546
 ],
 "worst_fold_ctrl": -0.00013928620773605857,
 "fold_table": [
  {
   "fold": 0,
   "thrC": 0.001,
   "thrF": 0.002,
   "nC": 3,
   "nF": 6,
   "oof_anchor": 0.6433363322169106,
   "oof_ctrl": 0.6433092796818337,
   "oof_full": 0.6428836841447984
  },
  {
   "fold": 1,
   "thrC": 0.0,
   "thrF": 0.002,
   "nC": 5,
   "nF": 4,
   "oof_anchor": 0.6566837940273769,
   "oof_ctrl": 0.6570826174046054,
   "oof_full": 0.6567426584